In [ ]:
import json
import yaml
from pathlib import Path
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
    WordFormatOption,
    ImageFormatOption
)
from docling.datamodel.pipeline_options import PdfPipelineOptions, EasyOcrOptions

from docling.pipeline.simple_pipeline import SimplePipeline
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline

In [11]:
# 모듈 최상단에 패턴 컴파일
import re
NEWLINE_PATTERN = re.compile(r'\r\n\d+')

def normalize_newlines(text: str) -> str:
    """개행문자 정규화 (동기 함수)"""
    return NEWLINE_PATTERN.sub('\n', text)

In [18]:
import os
from time import sleep, time
import pickle
import pdfplumber
from tqdm.auto import tqdm
from langchain_core.documents import Document


# 공유 가능한 옵션 정의
DEFAULT_PIPELINE_OPTIONS = PdfPipelineOptions(
    do_ocr=True,
    do_table_structure=True,
    ocr_options=EasyOcrOptions(lang=["en", "ko"])
    )


def parsing_pdf_by_page_with_docling(path:str, lv1_cat:str, lv2_cat:str):
    path = path.replace("\\", "/")
    filename = path.split("/")[-1]

    first_sentence = f"This page explains {filename.replace(".pdf", "")} that belongs to {lv1_cat} and  {lv2_cat} categories.\n"


    pipeline_options = DEFAULT_PIPELINE_OPTIONS
    converter = DocumentConverter(
        allowed_formats=[
            InputFormat.PDF
        ],
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options,
                backend=PyPdfiumDocumentBackend
            ),}
    )
    loaded_docs = converter.convert(path)
    with pdfplumber.open(path) as pdf:
        page_num = 0
        docs = []
        for _ in tqdm(pdf.pages):
            docling_text = loaded_docs.document.export_to_markdown(page_no=int(page_num)+1)
            docling_text = docling_text.replace("<!-- image -->", "")
            docling_text = normalize_newlines(docling_text)
            docling_text = first_sentence + docling_text
            lang_doc = Document(page_content=docling_text, metadata={'filename': filename, 'lv1_cat': lv1_cat, 'lv2_cat': lv2_cat, 'page':str(page_num)})
            docs.append(lang_doc)
            page_num+=1
            sleep(0.1)
    
    parsed_foldername = f"{lv1_cat}_{lv2_cat}"
    if not os.path.exists(f"../docs/{parsed_foldername}"):
        os.makedirs(f"../docs/{parsed_foldername}")
        
    parsed_filename = filename.replace(".pdf", "")
    with open(f"../docs/{parsed_foldername}/{parsed_filename}.pkl", 'ab') as file:
        pickle.dump(docs, file)

    # if os.path.exists(path):
    #     os.remove(path)

    return docs

In [19]:
start_time = time()
path = f"../data/Guidance for Noise and Vibration_2020.pdf"
lv1_cat, lv2_cat = "Rule", "KR"
result = parsing_pdf_by_page_with_docling(path=path, lv1_cat=lv1_cat, lv2_cat=lv2_cat)
end_time = time() - start_time
print(f"Document converted and tables exported in {end_time:.2f} seconds.")


2025-10-08 08:42:25,467 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-08 08:42:25,470 - INFO - Going to convert document batch...
2025-10-08 08:42:25,471 - INFO - Initializing pipeline for StandardPdfPipeline with options hash acb7af01651b138d096f2f800318bf95
2025-10-08 08:42:25,472 - INFO - Accelerator device: 'cpu'
2025-10-08 08:42:28,338 - INFO - Accelerator device: 'cpu'
2025-10-08 08:42:29,712 - INFO - Accelerator device: 'cpu'
2025-10-08 08:42:30,165 - INFO - Processing document Guidance for Noise and Vibration_2020.pdf
2025-10-08 08:43:27,685 - INFO - Finished converting document Guidance for Noise and Vibration_2020.pdf in 62.22 sec.
100%|██████████| 13/13 [00:01<00:00,  7.07it/s]

Document converted and tables exported in 64.12 seconds.


In [29]:
# Open the file containing the pickled data in binary read mode ('rb')
with open('../docs/Rule_KR/KR Notation Guide_2025.pkl', 'rb') as file:
    # Load the pickled object from the file
    loaded_object = pickle.load(file)

In [30]:
loaded_object[:5]

[Document(metadata={'filename': 'KR Notation Guide_2025.pdf', 'lv1_cat': 'RULE', 'lv2_cat': 'KR', 'page': '0'}, page_content='This page explains KR Notation Guide_2025 that belongs to RULE and  KR categories.\n\n\n## 2025\n\n## Notation Guide\n\n## KR'),
 Document(metadata={'filename': 'KR Notation Guide_2025.pdf', 'lv1_cat': 'RULE', 'lv2_cat': 'KR', 'page': '1'}, page_content='This page explains KR Notation Guide_2025 that belongs to RULE and  KR categories.\n## CONTENTS\n\n| CHAPTER 1                                                                                                                             | GENERAL ···········································································································  1                  |\n|---------------------------------------------------------------------------------------------------------------------------------------|----------------------------------------------------------------------------------------------------------

In [41]:
from IPython.display import Markdown
Markdown(loaded_object[75].page_content)

This page explains KR Notation Guide_2025 that belongs to RULE and  KR categories.
## 8-3 Oil/Liquefied Gas Carrier

|                                                                                           | Special Feature Notations                           | Special Feature Notations   | Special Feature Notations   | Special Feature Notations   | Special Feature Notations                                              | Special Feature Notations   | Special Feature Notations   | Special Feature Notations   | Special Feature Notations   | Special Feature Notations   |
|-------------------------------------------------------------------------------------------|-----------------------------------------------------|-----------------------------|-----------------------------|-----------------------------|------------------------------------------------------------------------|-----------------------------|-----------------------------|-----------------------------|-----------------------------|-----------------------------|
| Ship Type Notations                                                                       | Oil Tanker                                          | Liquefied Gas Carrier       | Liquefied Gas Carrier       | Liquefied Gas Carrier       | Liquefied Gas Carrier                                                  | Liquefied Gas Carrier       | Liquefied Gas Carrier       | Liquefied Gas Carrier       | Liquefied Gas Carrier       | Liquefied Gas Carrier       |
| Oil/Liquefied Gas Carrier 'ESP'  (Double Hull) (Double Hull)(EXP) (FAC) (FAO) (FBC) (CSR) | Crude Product Crude/Product Product/Asphalt Asphalt | A                           | B                           | (C)                         | Design Aspect and/or Primary Carg                                      | IMO Code                    | IMO Code                    | IMO Code                    | IMO Code                    | IMO Code                    |
|                                                                                           |                                                     | 1G 2G 2PG                   | 2I 3M 3S 1A                 | (R) (P) (RP)                | Maximum Vapour Pressure, Minimum Temperature and Specific Gravity (SG) | (NIGC) (IGC) (GC) (GCX)     | (NIGC) (IGC) (GC) (GCX)     | (NIGC) (IGC) (GC) (GCX)     | (NIGC) (IGC) (GC) (GCX)     | (NIGC) (IGC) (GC) (GCX)     |
|                                                                                           |                                                     | 3G                          |                             |                             |                                                                        |                             |                             |                             |                             |                             |
|                                                                                           |                                                     |                             | 1B                          |                             | Name of Liquefied Gas                                                  |                             |                             |                             |                             |                             |
|                                                                                           |                                                     |                             | 1C                          |                             | primarily carried                                                      |                             |                             |                             |                             |                             |
|                                                                                           |                                                     |                             | NV                          |                             |                                                                        |                             |                             |                             |                             |                             |



